In [2]:
import duckdb

con = duckdb.connect()

# Load SQLite extension
con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

# Attach source databases
con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2t.sqlite' AS t_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2d.sqlite' AS td_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_rh.sqlite' AS rh_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_vpd.sqlite' AS vpd_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_tp.sqlite' AS tp_db
""")

In [3]:
df = con.execute("""
SELECT
    t.datetime,
    t.point_id,

    -- T
    t."0"  AS t_0,
    t."3"  AS t_3,
    t."6"  AS t_6,
    t."9"  AS t_9,
    t."12" AS t_12,
    t."15" AS t_15,
    t."18" AS t_18,
    t."21" AS t_21,

    -- Td
    td."0"  AS td_0,
    td."3"  AS td_3,
    td."6"  AS td_6,
    td."9"  AS td_9,
    td."12" AS td_12,
    td."15" AS td_15,
    td."18" AS td_18,
    td."21" AS td_21,

    -- RH
    rh."0"  AS rh_0,
    rh."3"  AS rh_3,
    rh."6"  AS rh_6,
    rh."9"  AS rh_9,
    rh."12" AS rh_12,
    rh."15" AS rh_15,
    rh."18" AS rh_18,
    rh."21" AS rh_21,

    -- VPD
    vpd."0"  AS vpd_0,
    vpd."3"  AS vpd_3,
    vpd."6"  AS vpd_6,
    vpd."9"  AS vpd_9,
    vpd."12" AS vpd_12,
    vpd."15" AS vpd_15,
    vpd."18" AS vpd_18,
    vpd."21" AS vpd_21,

    -- TP (precip)
    tp."0"  AS tp_0,
    tp."3"  AS tp_3,
    tp."6"  AS tp_6,
    tp."9"  AS tp_9,
    tp."12" AS tp_12,
    tp."15" AS tp_15,
    tp."18" AS tp_18,
    tp."21" AS tp_21

FROM t_db.daily_data t

JOIN td_db.daily_data td
    ON t.datetime = td.datetime
    AND t.point_id = td.point_id

JOIN rh_db.daily_data rh
    ON t.datetime = rh.datetime
    AND t.point_id = rh.point_id

JOIN vpd_db.daily_data vpd
    ON t.datetime = vpd.datetime
    AND t.point_id = vpd.point_id

JOIN tp_db.daily_data tp
    ON t.datetime = tp.datetime
    AND t.point_id = tp.point_id

LIMIT 10
""").df()

In [4]:
df.shape

(10, 42)

In [5]:
df.head()

,datetime,point_id,t_0,t_3,t_6,t_9,t_12,t_15,t_18,t_21,...,vpd_18,vpd_21,tp_0,tp_3,tp_6,tp_9,tp_12,tp_15,tp_18,tp_21
0,19900210,957,267.065430,263.428711,264.655869,265.572342,265.585754,266.067200,269.377197,271.200195,...,0.172037,0.204872,0.000004,0.000000,0.000000,0.000024,9.641349e-05,0.000165,0.000230,0.000268
1,19900211,957,270.001953,268.289795,268.919189,267.946777,267.359802,266.892181,274.045654,277.164062,...,0.336172,0.481390,0.000294,0.000021,0.000047,0.000066,8.513927e-05,0.000091,0.000091,0.000091
2,19900212,957,273.262207,267.403809,267.093018,266.308105,264.270264,265.149658,276.031982,277.797363,...,0.423487,0.483175,0.000091,0.000000,0.000000,0.000000,4.261732e-07,0.000001,0.000001,0.000003
3,19900213,957,274.595459,269.667236,269.264893,267.130859,266.377686,264.063232,267.954102,269.920166,...,0.146751,0.199410,0.000013,0.000121,0.000547,0.001048,1.205425e-03,0.001221,0.001630,0.001754
4,19900214,957,265.542725,259.716309,257.737061,259.564850,260.016586,258.978409,265.255356,264.961182,...,0.109691,0.105128,0.002368,0.000468,0.000912,0.001115,1.255453e-03,0.001343,0.001458,0.001926


In [8]:
con.execute("""
SELECT
    t.point_id,

    SUBSTR(t.datetime, 1, 7) AS year_month,

    (
        SUM(
            rh."0" + rh."3" + rh."6" + rh."9" +
            rh."12" + rh."15" + rh."18" + rh."21"
        )
        /
        (COUNT(*) * 8)
    ) AS rh_mean

FROM t_db.daily_data t

JOIN rh_db.daily_data rh
    ON t.datetime = rh.datetime
    AND t.point_id = rh.point_id

GROUP BY
    t.point_id,
    year_month

ORDER BY
    t.point_id,
    year_month

LIMIT 20
""").df()

,point_id,year_month,rh_mean
0,0,1990010,73.750374
1,0,1990011,79.845200
2,0,1990012,68.556329
3,0,1990013,76.284280
4,0,1990020,78.674196
5,0,1990021,79.029405
6,0,1990022,70.869412
7,0,1990030,88.032993
8,0,1990031,74.830698
9,0,1990032,69.930898


In [9]:
con.execute("""
SELECT
    t.point_id,

    SUBSTR(t.datetime, 1, 7) AS year_month,

    -- T mean
    SUM(
        t."0" + t."3" + t."6" + t."9" +
        t."12" + t."15" + t."18" + t."21"
    ) / (COUNT(*) * 8) AS t_mean,

    -- Td mean
    SUM(
        td."0" + td."3" + td."6" + td."9" +
        td."12" + td."15" + td."18" + td."21"
    ) / (COUNT(*) * 8) AS td_mean,

    -- RH mean
    SUM(
        rh."0" + rh."3" + rh."6" + rh."9" +
        rh."12" + rh."15" + rh."18" + rh."21"
    ) / (COUNT(*) * 8) AS rh_mean,

    -- VPD mean
    SUM(
        vpd."0" + vpd."3" + vpd."6" + vpd."9" +
        vpd."12" + vpd."15" + vpd."18" + vpd."21"
    ) / (COUNT(*) * 8) AS vpd_mean,

    -- TP monthly sum (NO DIVISION)
    SUM(
        tp."0" + tp."3" + tp."6" + tp."9" +
        tp."12" + tp."15" + tp."18" + tp."21"
    ) AS tp_sum

FROM t_db.daily_data t

JOIN td_db.daily_data td
    ON t.datetime = td.datetime
    AND t.point_id = td.point_id

JOIN rh_db.daily_data rh
    ON t.datetime = rh.datetime
    AND t.point_id = rh.point_id

JOIN vpd_db.daily_data vpd
    ON t.datetime = vpd.datetime
    AND t.point_id = vpd.point_id

JOIN tp_db.daily_data tp
    ON t.datetime = tp.datetime
    AND t.point_id = tp.point_id

GROUP BY
    t.point_id,
    year_month

ORDER BY
    t.point_id,
    year_month

LIMIT 20
""").df()

,point_id,year_month,t_mean,td_mean,rh_mean,vpd_mean,tp_sum
0,0,1990010,264.079891,260.165411,73.750374,0.088783,0.023216
1,0,1990011,267.179343,264.054605,79.845200,0.096466,0.026033
2,0,1990012,262.753003,257.996721,68.556329,0.093833,0.033747
3,0,1990013,264.472836,260.965568,76.284280,0.081586,0.008594
4,0,1990020,262.789498,259.728597,78.674196,0.063023,0.046106
5,0,1990021,262.408125,259.437601,79.029405,0.059123,0.134972
6,0,1990022,266.724186,262.115857,70.869412,0.119998,0.001652
7,0,1990030,267.852741,266.144295,88.032993,0.054542,0.041170
8,0,1990031,267.738444,263.857241,74.830698,0.108662,0.060754
9,0,1990032,272.676235,267.664045,69.930898,0.194228,0.037153
